# 03 — Dataset Evaluation

This notebook evaluates the live autoencoder's detection performance against
labeled IoT cybersecurity datasets.

**Supported datasets** (from `ML_IOT_PART/DATASETS.md`):
- CIC-IoT-2022
- BoT-IoT
- Edge-IIoT
- IoT-NID
- Kitsune

If you don't have any of these downloaded, this notebook will generate
synthetic labeled data to demonstrate the evaluation pipeline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'python-backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0e14'
matplotlib.rcParams['axes.facecolor'] = '#12161f'
matplotlib.rcParams['text.color'] = '#e6edf3'
matplotlib.rcParams['axes.labelcolor'] = '#8b949e'
matplotlib.rcParams['xtick.color'] = '#8b949e'
matplotlib.rcParams['ytick.color'] = '#8b949e'

from ml_engine import MLEngine, FEATURE_NAMES, FEATURE_DIM

In [ ]:
MODEL_PATH = os.path.join('..', 'python-backend', 'model.pkl')
engine = MLEngine(input_dim=FEATURE_DIM)

if os.path.exists(MODEL_PATH):
    engine.load(MODEL_PATH)
    print(f'Loaded model. Status: {engine.get_status()}')
else:
    # Train on synthetic normal data if no model exists
    print('No saved model found. Training on synthetic normal data...')
    rng = np.random.default_rng(42)
    center = rng.uniform(10, 500, size=FEATURE_DIM)
    for _ in range(600):
        sample = center + rng.normal(0, center * 0.1, size=FEATURE_DIM)
        sample = np.clip(sample, 0, None)
        engine.train_packet(sample)
    engine.finalize_training()
    print(f'Synthetic model trained. Status: {engine.get_status()}')

## Generate Labeled Evaluation Data

Since the actual IoT datasets are very large, we generate synthetic labeled data
that mimics the statistical properties of normal vs attack traffic.

In [ ]:
rng = np.random.default_rng(123)

n_normal = 500
n_attack = 100

# Normal traffic: close to the scaler's learned mean
if engine._scaler._fitted:
    mean = engine._scaler.mean_
    std = engine._scaler.std_
else:
    mean = np.ones(FEATURE_DIM) * 100
    std = np.ones(FEATURE_DIM) * 20

normal_data = mean + rng.normal(0, std * 0.5, size=(n_normal, FEATURE_DIM))
normal_data = np.clip(normal_data, 0, None)
normal_labels = np.zeros(n_normal)  # 0 = normal

# Attack traffic: shifted significantly
attack_data = mean + rng.normal(0, std * 3.0, size=(n_attack, FEATURE_DIM))
attack_data += std * 2.0  # Add a bias shift
attack_data = np.clip(attack_data, 0, None)
attack_labels = np.ones(n_attack)  # 1 = attack

# Combine
X = np.vstack([normal_data, attack_data])
y_true = np.concatenate([normal_labels, attack_labels])

print(f'Evaluation dataset: {n_normal} normal + {n_attack} attack = {len(X)} total')

## Score All Samples & Compute Metrics

In [ ]:
scores = np.array([engine.score_packet(x) for x in X])
predictions = np.array([engine.predict_packet(x) for x in X]).astype(int)

# Confusion matrix components
TP = np.sum((predictions == 1) & (y_true == 1))
FP = np.sum((predictions == 1) & (y_true == 0))
TN = np.sum((predictions == 0) & (y_true == 0))
FN = np.sum((predictions == 0) & (y_true == 1))

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (TP + TN) / len(y_true)

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')
print(f'\nConfusion Matrix:')
print(f'  TP={TP}  FP={FP}')
print(f'  FN={FN}  TN={TN}')

## Score Distribution Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(scores[y_true == 0], bins=50, alpha=0.7, color='#3fb950', label='Normal', density=True)
ax.hist(scores[y_true == 1], bins=30, alpha=0.7, color='#f85149', label='Attack', density=True)

if engine._threshold:
    ax.axvline(engine._threshold, color='#f0883e', linestyle='--', linewidth=2, 
               label=f'Threshold ({engine._threshold:.4f})')

ax.set_xlabel('Anomaly Score (Reconstruction Error)')
ax.set_ylabel('Density')
ax.set_title('Anomaly Score Distribution: Normal vs Attack', fontsize=14, pad=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## ROC Curve

In [ ]:
# Manual ROC computation (no sklearn dependency)
thresholds = np.linspace(scores.min(), scores.max(), 200)
tpr_list = []
fpr_list = []

for t in thresholds:
    tp = np.sum((scores > t) & (y_true == 1))
    fp = np.sum((scores > t) & (y_true == 0))
    fn = np.sum((scores <= t) & (y_true == 1))
    tn = np.sum((scores <= t) & (y_true == 0))
    tpr_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0)

# AUC (trapezoid rule)
sorted_pairs = sorted(zip(fpr_list, tpr_list))
fpr_sorted = [p[0] for p in sorted_pairs]
tpr_sorted = [p[1] for p in sorted_pairs]
auc = np.trapz(tpr_sorted, fpr_sorted)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr_list, tpr_list, color='#58a6ff', linewidth=2, label=f'ROC Curve (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], color='#484f58', linestyle='--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Autoencoder Anomaly Detector', fontsize=14, pad=12)
ax.legend(fontsize=11)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

print(f'AUC-ROC: {auc:.4f}')